# FoundryIQ Docs → Azure AI Search Index

This notebook:
1. **Creates** an Azure AI Search index with vector + semantic search support
2. **Reads** the three Markdown documents in `foundryiq-data/`
3. **Generates embeddings** via Azure OpenAI (text-embedding-3-small)
4. **Uploads** all documents to the index

**Authentication:** Entra ID (`DefaultAzureCredential`) — no API keys required for AAIS.

> **Pre-requisite:** Run `az login` (or ensure a Managed Identity / Service Principal is active) so `DefaultAzureCredential` can resolve a token.

In [1]:
# Install required packages
%pip install python-dotenv azure-search-documents==11.6.0b7 azure-identity openai --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\lananoor\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import uuid
from pathlib import Path
from dotenv import load_dotenv

# Azure SDK imports
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    SearchIndex,
)
from openai import AzureOpenAI

# Load .env
load_dotenv(override=True)
print("✅ Packages loaded")

✅ Packages loaded


## 1. Configuration

All values are read from `.env`.  
**No API key is used for Azure AI Search** — only `AAIS_ENDPOINT` is needed.  
For Azure OpenAI, the endpoint and embedding deployment name are required.

In [ ]:
# ── Azure AI Search ──────────────────────────────────────────────────────────
AAIS_ENDPOINT = os.getenv("AAIS_ENDPOINT")
INDEX_NAME    = os.getenv("AAIS_INDEX_NAME", "budget-reports-workflow-index")

# ── Azure OpenAI ─────────────────────────────────────────────────────────────
# Inferred from existing project config; override via .env if needed
AOAI_ENDPOINT          = os.getenv("AOAI_ENDPOINT")
EMBEDDING_DEPLOYMENT   = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
EMBEDDING_DIMENSIONS   = 1536  # text-embedding-3-small output size

# ── Data source ───────────────────────────────────────────────────────────────
# Path is relative to this notebook; adjust if you move the file
DOCS_FOLDER = Path("foundryiq-data")

print(f"AAIS endpoint  : {AAIS_ENDPOINT}")
print(f"Index name     : {INDEX_NAME}")
print(f"AOAI endpoint  : {AOAI_ENDPOINT}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT}")
print(f"Docs folder    : {DOCS_FOLDER.resolve()}")

## 2. Entra ID Authentication

`DefaultAzureCredential` tries the following in order:
1. Environment variables (`AZURE_CLIENT_ID`, `AZURE_TENANT_ID`, `AZURE_CLIENT_SECRET`)
2. Workload Identity (AKS)
3. Managed Identity
4. Azure CLI (`az login`)
5. Azure Developer CLI (`azd auth login`)
6. Visual Studio Code / Visual Studio

Run `az login` first if you are working locally.

In [20]:
# Single credential object reused for both AAIS and AOAI
credential = DefaultAzureCredential()

# ── Azure AI Search clients (Entra ID) ───────────────────────────────────────
index_client  = SearchIndexClient(endpoint=AAIS_ENDPOINT, credential=credential)
search_client = SearchClient(endpoint=AAIS_ENDPOINT, index_name=INDEX_NAME, credential=credential)

# ── Azure OpenAI client (Entra ID token provider) ────────────────────────────
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

openai_client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01",
)

print("✅ Entra ID credential initialised")
print("✅ SearchIndexClient, SearchClient, and AzureOpenAI ready")

✅ Entra ID credential initialised
✅ SearchIndexClient, SearchClient, and AzureOpenAI ready


## 3. Create the Search Index

The index schema includes:
- `id` — unique document key
- `title` — document / section title
- `content` — full text chunk (searchable)
- `source` — original filename
- `chunk_index` — position within the source file
- `content_vector` — embedding for vector similarity search

Vector search uses **HNSW** with cosine similarity, and semantic search is also configured.

In [7]:
def create_index():
    fields = [
        SimpleField(
            name="id",
            type=SearchFieldDataType.String,
            key=True,
            filterable=True,
        ),
        SearchableField(
            name="title",
            type=SearchFieldDataType.String,
            filterable=True,
            sortable=True,
        ),
        SearchableField(
            name="content",
            type=SearchFieldDataType.String,
            analyzer_name="en.microsoft",
        ),
        SimpleField(
            name="source",
            type=SearchFieldDataType.String,
            filterable=True,
            facetable=True,
        ),
        SimpleField(
            name="chunk_index",
            type=SearchFieldDataType.Int32,
            filterable=True,
            sortable=True,
        ),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=EMBEDDING_DIMENSIONS,
            vector_search_profile_name="foundryiq-vector-profile",
        ),
    ]

    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="foundryiq-hnsw",
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="foundryiq-vector-profile",
                algorithm_configuration_name="foundryiq-hnsw",
            )
        ],
    )

    semantic_config = SemanticConfiguration(
        name="foundryiq-semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="content")],
            keywords_fields=[SemanticField(field_name="source")],
        ),
    )

    index = SearchIndex(
        name=INDEX_NAME,
        fields=fields,
        vector_search=vector_search,
        semantic_search=SemanticSearch(configurations=[semantic_config]),
    )

    result = index_client.create_or_update_index(index)
    print(f"✅ Index '{result.name}' created / updated successfully")
    return result

create_index()

✅ Index 'budget-reports-workflow-index' created / updated successfully


## 4. Read & Chunk the Markdown Files

Each file is split into chunks at heading boundaries (`## `) so that each section becomes a discrete document in the index. A fallback character-level chunker handles very long sections.

In [17]:
def split_markdown(text: str, max_chars: int = 2000) -> list[dict]:
    """Split markdown on H2 headings; further split long chunks by paragraph."""
    import re

    # Split on level-2 headings (## ...)
    sections = re.split(r"(?m)^(## .+)", text)

    chunks = []
    current_title = "Introduction"
    current_body  = ""

    for part in sections:
        if re.match(r"^## .+", part):
            # Flush previous section
            if current_body.strip():
                chunks.append({"title": current_title, "content": current_body.strip()})
            current_title = part.strip("# ").strip()
            current_body  = part + "\n"
        else:
            current_body += part

    # Flush last section
    if current_body.strip():
        chunks.append({"title": current_title, "content": current_body.strip()})

    # Sub-split any chunks exceeding max_chars by paragraph
    final_chunks = []
    for chunk in chunks:
        if len(chunk["content"]) <= max_chars:
            final_chunks.append(chunk)
        else:
            paragraphs = chunk["content"].split("\n\n")
            buffer = ""
            sub_idx = 0
            for para in paragraphs:
                if len(buffer) + len(para) > max_chars and buffer:
                    final_chunks.append({
                        "title": f"{chunk['title']} (part {sub_idx + 1})",
                        "content": buffer.strip()
                    })
                    sub_idx += 1
                    buffer = para + "\n\n"
                else:
                    buffer += para + "\n\n"
            if buffer.strip():
                final_chunks.append({
                    "title": f"{chunk['title']} (part {sub_idx + 1})" if sub_idx > 0 else chunk["title"],
                    "content": buffer.strip()
                })

    return final_chunks


def load_documents(docs_folder: Path) -> list[dict]:
    """Read all .md files and produce a flat list of chunk dicts."""
    all_docs = []
    md_files = sorted(docs_folder.glob("*.md"))

    if not md_files:
        raise FileNotFoundError(f"No .md files found in {docs_folder.resolve()}")

    for md_file in md_files:
        print(f"📄 Reading: {md_file.name}")
        text   = md_file.read_text(encoding="utf-8")
        chunks = split_markdown(text)

        for idx, chunk in enumerate(chunks):
            doc_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{md_file.name}:{idx}"))
            all_docs.append({
                "id":          doc_id,
                "title":       chunk["title"],
                "content":     chunk["content"],
                "source":      md_file.name,
                "chunk_index": idx,
            })

        print(f"   → {len(chunks)} chunks created")

    print(f"\n✅ Total chunks across all files: {len(all_docs)}")
    return all_docs


documents = load_documents(DOCS_FOLDER)
# Preview first chunk
print("\n--- First chunk preview ---")
print(json.dumps({k: v for k, v in documents[0].items() if k != "content_vector"}, indent=2, ensure_ascii=False)[:600])

📄 Reading: ADGA_Financial_Management_Act_2024.md
   → 9 chunks created
📄 Reading: IT_Technology_Spending_Guidelines_2026.md
   → 12 chunks created
📄 Reading: Public_Sector_Procurement_Guidelines_2026.md
   → 11 chunks created

✅ Total chunks across all files: 32

--- First chunk preview ---
{
  "id": "b309cad1-2109-580b-88f9-8f49c9b0a6c5",
  "title": "Introduction",
  "content": "# Apex Digital Government Authority",
  "source": "ADGA_Financial_Management_Act_2024.md",
  "chunk_index": 0
}


## 5. Generate Embeddings (Entra ID)

Embeddings are generated in small batches to stay within the Azure OpenAI rate limits.

In [21]:
def generate_embeddings(docs: list[dict], batch_size: int = 16) -> list[dict]:
    """Add 'content_vector' to each doc using Azure OpenAI Embeddings (Entra ID)."""
    total   = len(docs)
    updated = []

    for start in range(0, total, batch_size):
        batch   = docs[start : start + batch_size]
        texts   = [d["content"] for d in batch]

        response = openai_client.embeddings.create(
            model=EMBEDDING_DEPLOYMENT,
            input=texts,
        )

        for doc, embedding_obj in zip(batch, response.data):
            doc["content_vector"] = embedding_obj.embedding
            updated.append(doc)

        end = min(start + batch_size, total)
        print(f"   Embedded chunks {start + 1}–{end} / {total}")

    print(f"\n✅ Embeddings generated for all {total} chunks")
    return updated


print("🔄 Generating embeddings via Azure OpenAI (Entra ID)...")
documents = generate_embeddings(documents)

🔄 Generating embeddings via Azure OpenAI (Entra ID)...
   Embedded chunks 1–16 / 32
   Embedded chunks 17–32 / 32

✅ Embeddings generated for all 32 chunks


## 6. Upload Documents to the Index

In [22]:
def upload_documents(docs: list[dict], batch_size: int = 50) -> None:
    """Upload documents in batches using merge-or-upload (idempotent)."""
    total = len(docs)

    for start in range(0, total, batch_size):
        batch  = docs[start : start + batch_size]
        result = search_client.merge_or_upload_documents(batch)

        succeeded = sum(1 for r in result if r.succeeded)
        failed    = sum(1 for r in result if not r.succeeded)
        end       = min(start + batch_size, total)

        status = f"✅ Batch {start + 1}–{end}: {succeeded} succeeded"
        if failed:
            status += f", ⚠️ {failed} failed"
        print(status)

    print(f"\n🎉 Upload complete — {total} chunks pushed to index '{INDEX_NAME}'")


print(f"🔄 Uploading {len(documents)} chunks to Azure AI Search...")
upload_documents(documents)

🔄 Uploading 32 chunks to Azure AI Search...
✅ Batch 1–32: 32 succeeded

🎉 Upload complete — 32 chunks pushed to index 'budget-reports-workflow-index'


## 7. Verify — Quick Test Query

In [23]:
from azure.search.documents.models import VectorizedQuery

def test_search(query_text: str, top: int = 3):
    """Run a hybrid (keyword + vector) search against the index."""
    # Generate embedding for the query
    response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=[query_text],
    )
    query_vector = response.data[0].embedding

    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top,
        fields="content_vector",
    )

    results = search_client.search(
        search_text=query_text,
        vector_queries=[vector_query],
        select=["id", "title", "source", "chunk_index"],
        top=top,
    )

    print(f"Query: \"{query_text}\"\n")
    for r in results:
        print(f"  [{r['source']}]  {r['title']}  (chunk {r['chunk_index']})")
        print(f"  Score: {r['@search.score']:.4f}")
        print()

# --- Run sample queries ---
test_search("budget variance thresholds and escalation procedures")
test_search("IT procurement approval levels")
test_search("emergency procurement exceptions")

Query: "budget variance thresholds and escalation procedures"

  [ADGA_Financial_Management_Act_2024.md]  PART III: VARIANCE THRESHOLDS AND ESCALATION  (chunk 4)
  Score: 0.0333

  [IT_Technology_Spending_Guidelines_2026.md]  8. VARIANCE ANALYSIS FOR IT SPENDING  (chunk 9)
  Score: 0.0328

  [Public_Sector_Procurement_Guidelines_2026.md]  5. VARIANCE IMPACT ON PROCUREMENT  (chunk 5)
  Score: 0.0311

Query: "IT procurement approval levels"

  [Public_Sector_Procurement_Guidelines_2026.md]  2. PROCUREMENT THRESHOLDS  (chunk 2)
  Score: 0.0331

  [Public_Sector_Procurement_Guidelines_2026.md]  3. IT & TECHNOLOGY PROCUREMENT  (chunk 3)
  Score: 0.0323

  [Public_Sector_Procurement_Guidelines_2026.md]  9. SPECIAL PROVISIONS FOR DIGITAL TRANSFORMATION  (chunk 9)
  Score: 0.0300

Query: "emergency procurement exceptions"

  [Public_Sector_Procurement_Guidelines_2026.md]  2. PROCUREMENT THRESHOLDS  (chunk 2)
  Score: 0.0333

  [ADGA_Financial_Management_Act_2024.md]  PART IV: PROCUREMENT AND S

---
## Summary

| Step | Detail |
|------|--------|
| Index | `foundryiq-docs-index` on `agentssearchservice.search.windows.net` |
| Auth | Entra ID via `DefaultAzureCredential` (no API keys) |
| Embedding model | `text-embedding-3-small` (1536 dims) |
| Chunking | Markdown H2-section split, max 2 000 chars |
| Source files | `ADGA_Financial_Management_Act_2024.md`, `IT_Technology_Spending_Guidelines_2026.md`, `Public_Sector_Procurement_Guidelines_2026.md` |

The index is now ready to be used as a knowledge source in FoundryIQ or any Azure AI Foundry agent.